# Day 3 — Retrieval-Augmented Generation (RAG) Exercises
## Industrial AI & LLM Training Program

**Session:** Day 3 of 5 — RAG Systems  
**Time:** 20:00–22:30 WIB  
**Prerequisites:** Day 1 (text mining), Day 2 (LLM API & prompting)

---

### What you will build tonight

| Section | Topic | Output |
|---|---|---|
| 0 | Setup & provider detection | Working environment |
| 1 | The Problem — LLM without memory | Hallucination example |
| 2 | Document loading & chunking | 30-doc knowledge base, chunked |
| 3 | Embeddings | Embedding matrix, cosine similarity |
| 4 | Vector databases: FAISS & Chroma | Two indexed knowledge bases |
| 5 | Context injection & RAG query | Grounded LLM answers |
| 6 | Full RAG pipeline — all 40 tickets | `pipeline_df` DataFrame |
| 7 | RAG evaluation | Faithfulness, groundedness, latency |
| 8 | Reflection & Day 4 preview | — |

---

> **Cross-day anchors:**
> - Day 1: K-Means silhouette = 0.027 (vocabulary overlap problem)
> - Day 2: LLM triage achieved 90%+ accuracy on the 40-ticket dataset
> - Day 3 (tonight): LLM + RAG knowledge base → grounded, verifiable answers
> - Day 4 (next): LangChain RAG chains, LangGraph multi-step retrieval agent


In [ ]:
# CELL 0-A: Install required libraries (run this if you get ImportError below)
# Uncomment and run if needed:

# !pip install anthropic openai requests numpy pandas matplotlib
# !pip install sentence-transformers faiss-cpu chromadb tiktoken python-dotenv


In [1]:
# CELL 0-B: Imports and provider detection
import os, json, time, warnings, re
import numpy as np
import pandas as pd
import requests
warnings.filterwarnings('ignore')

try:
    from dotenv import load_dotenv
    if load_dotenv():
        print('✓ .env file loaded')
    else:
        print('  (No .env file found)')
except ImportError:
    print('  (python-dotenv not installed)')

PROVIDER = None
client = None
DEFAULT_MODEL = None

if os.environ.get('ANTHROPIC_API_KEY'):
    try:
        import anthropic
        client = anthropic.Anthropic()
        PROVIDER = 'anthropic'
        DEFAULT_MODEL = 'claude-haiku-4-5-20251001'
        print(f'✓ Provider: Anthropic  |  Model: {DEFAULT_MODEL}')
    except Exception as e:
        print(f'Anthropic import failed: {e}')

if PROVIDER is None and os.environ.get('OPENAI_API_KEY'):
    try:
        import openai
        client = openai.OpenAI()
        PROVIDER = 'openai'
        DEFAULT_MODEL = 'gpt-3.5-turbo'
        print(f'✓ Provider: OpenAI  |  Model: {DEFAULT_MODEL}')
    except Exception as e:
        print(f'OpenAI import failed: {e}')

if PROVIDER is None:
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            PROVIDER = 'ollama'
            DEFAULT_MODEL = 'llama3.2'
            print(f'✓ Provider: Ollama  |  Model: {DEFAULT_MODEL}')
    except Exception:
        pass

if PROVIDER is None:
    PROVIDER = 'mock'
    DEFAULT_MODEL = 'mock-rag-v1'
    print('⚠  Provider: Mock (offline) | Model: mock-rag-v1')
    print('   All LLM calls return deterministic mock responses.')
    print('   Set ANTHROPIC_API_KEY or start Ollama to use a real model.')

EMBED_PROVIDER = 'mock'
try:
    from sentence_transformers import SentenceTransformer
    EMBED_MODEL = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    EMBED_PROVIDER = 'sentence_transformers'
    print('✓ Embedding: sentence-transformers (paraphrase-multilingual-MiniLM-L12-v2)')
except ImportError:
    print('⚠  sentence-transformers not installed → using mock embeddings')
    print('   Install: pip install sentence-transformers')

print(f'Ready! Provider={PROVIDER}, Embed={EMBED_PROVIDER}')


✓ .env file loaded
✓ Provider: Anthropic  |  Model: claude-haiku-4-5-20251001


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3324.85it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Embedding: sentence-transformers (paraphrase-multilingual-MiniLM-L12-v2)
Ready! Provider=anthropic, Embed=sentence_transformers


---
## Section 1 — The Problem: LLM Without Memory

**Day 2 recap:** Our LLM achieved 90%+ accuracy categorizing the 40 industrial tickets.

**But what happens when we ask something NOT in the training data?**

Imagine a new technician asks: *"What is the exact LOTO procedure for centrifugal pump P-101?"*

- The LLM knows *what* LOTO is (general concept from training)
- The LLM does NOT know your plant's **specific** P-101 isolation points, tag numbers,
  lockout sequence, or responsible personnel

Without plant SOPs, the LLM will either:
1. **Hallucinate** — confidently invent plausible-sounding but wrong procedures
2. **Refuse** — say "I don't have access to your specific procedures"

Neither is acceptable in a safety-critical industrial environment.

**The RAG solution:** Give the LLM the right document BEFORE it answers.

```
Query → Retrieve relevant SOP → Inject into prompt → LLM answers from document
```


In [2]:
# CELL 1-A: Demonstrate LLM hallucination without a knowledge base

def _mock_llm_rag(messages, system='', context_chunks=None):
    user_text = ' '.join(m.get('content', '') for m in messages).lower()

    if context_chunks is None:
        # No KB - hallucination mode
        if 'p-101' in user_text and 'loto' in user_text:
            return (
                "[MOCK - no KB] The LOTO procedure for centrifugal pump P-101: "
                "(1) Notify shift supervisor, (2) Close suction valve V-001 and discharge valve V-002, "
                "(3) De-energize motor at MCC panel 3B, apply personal lock #47, "
                "(4) Bleed pressure via drain valve DV-103, (5) Verify zero energy. "
                "WARNING: HALLUCINATED - tag numbers, panel locations, and valve IDs were invented."
            )
        if 'me21n' in user_text:
            return (
                "[MOCK - no KB] In ME21N: enter vendor number, material code, quantity. "
                "System auto-populates pricing from info record. "
                "This may not match your SAP configuration."
            )
        if 'vpn' in user_text or 'scada' in user_text:
            return (
                "[MOCK - no KB] For SCADA connectivity: check network switch port status, "
                "verify PLC IP configuration, restart SCADA client service. "
                "Specific IPs and procedures depend on your site."
            )
        return (
            "[MOCK - no KB] I can provide general guidance but do not have access "
            "to your specific plant documentation. Please consult local procedures."
        )

    # With RAG - grounded answer
    context_text = ''.join(context_chunks).lower()
    if 'p-101' in user_text or 'loto' in user_text:
        if 'suction' in context_text or 'isolation' in context_text:
            return (
                "Based on SOP-MECH-001, the LOTO procedure for pump P-101: "
                "(1) Inform control room and get work permit, "
                "(2) Close suction and discharge valves, "
                "(3) De-energize motor at local MCC, apply LOTO lock and tag, "
                "(4) Depressurize via drain connection, "
                "(5) Verify zero energy state before work begins. "
                "[Source: SOP-MECH-001 Lockout/Tagout Procedure]"
            )
    if 'me21n' in user_text:
        return (
            "In ME21N, enter the vendor number, select purchasing organization, "
            "add line items with material number, quantity, and delivery date. "
            "System validates vendor master data before saving. "
            "[Source: SAP-GUIDE-001 ME21N Purchase Order Creation]"
        )
    return (
        f"[MOCK - with KB] Based on retrieved context ({len(context_chunks)} chunks), "
        "this response is grounded in your plant's actual documentation."
    )


def chat(messages, system=None, model=None, temperature=0.0, _context_chunks=None):
    model = model or DEFAULT_MODEL
    if PROVIDER == 'anthropic':
        kwargs = dict(model=model, max_tokens=1024, messages=messages, temperature=temperature)
        if system:
            kwargs['system'] = system
        return client.messages.create(**kwargs).content[0].text
    elif PROVIDER == 'openai':
        all_msgs = []
        if system:
            all_msgs.append({'role': 'system', 'content': system})
        all_msgs.extend(messages)
        return client.chat.completions.create(model=model, messages=all_msgs,
               temperature=temperature, max_tokens=1024).choices[0].message.content
    elif PROVIDER == 'ollama':
        all_msgs = []
        if system:
            all_msgs.append({'role': 'system', 'content': system})
        all_msgs.extend(messages)
        r = requests.post('http://localhost:11434/api/chat',
                          json={'model': model, 'messages': all_msgs, 'stream': False,
                                'options': {'temperature': temperature}}, timeout=60)
        r.raise_for_status()
        return r.json()['message']['content']
    elif PROVIDER == 'mock':
        return _mock_llm_rag(messages, system or '', _context_chunks)
    else:
        raise RuntimeError('No provider configured.')


print('=' * 70)
print('QUERY: What is the LOTO procedure for centrifugal pump P-101?')
print('CONTEXT: None (no knowledge base)')
print('=' * 70)

t0 = time.time()
response_no_rag = chat(
    messages=[{'role': 'user',
               'content': 'What is the LOTO procedure for centrifugal pump P-101? Exact steps.'}],
    system='You are an industrial maintenance assistant.',
)
latency_no_rag = time.time() - t0

print(f'LLM Response (no RAG):{response_no_rag}')
print(f'Latency: {latency_no_rag:.2f}s')
print('⚠  NOTICE: Response may contain invented tag numbers and valve IDs.')
print('   This is hallucination - do not use for actual maintenance.')


QUERY: What is the LOTO procedure for centrifugal pump P-101?
CONTEXT: None (no knowledge base)
LLM Response (no RAG):# LOTO Procedure for Centrifugal Pump P-101

I don't have access to your facility's specific equipment documentation or LOTO procedures. **You must refer to your facility's official lockout/tagout program** for P-101, as procedures vary by:

- Equipment design and manufacturer
- Facility standards
- Regulatory requirements (OSHA 1910.147)
- Energy sources present

## General LOTO Steps (Reference Only)

1. **Notify** affected personnel
2. **Locate** all energy sources (electrical, steam, hydraulic, etc.)
3. **Shut DOWN** the pump via normal controls
4. **ISOLATE** energy sources:
   - Open disconnect switches
   - Close isolation valves
   - Block/bleed pressurized lines
5. **DISSIPATE** residual energy (drain, depressurize, cool)
6. **APPLY** locks and tags to each energy source
7. **VERIFY** isolation (attempt startup, test gauges)
8. **PERFORM** maintenance work
9. *

---
## Section 2 — Document Loading & Chunking

### Why chunking matters

LLMs have context window limits. A typical industrial SOP is 2–10 pages (~500–3000 words).
If we inject a full 3000-word document into every query, we:
- Waste tokens (most content is irrelevant to the specific question)
- Hit context limits on longer documents
- Dilute the relevant content with noise

**Solution: chunking** — split documents into smaller, overlapping segments,
then retrieve only the most relevant chunks.

### Chunking strategies

| Strategy | How it works | Best for |
|---|---|---|
| **Fixed-size** | Split every N tokens/chars, with overlap | Quick prototyping |
| **Sentence-aware** | Split at sentence boundaries | Readable chunks |
| **Semantic** | Split at topic shifts (embedding distance) | Long docs with sections |
| **Hierarchical** | Chunk + summary at multiple levels | Multi-granularity retrieval |

**Tonight:** Fixed-size and sentence-aware chunking.

### Overlap

```
Chunk 1: [...text A...][...text B (overlap)...]
Chunk 2:               [...text B (overlap)...][...text C...]
```

Typical overlap: 10–20% of chunk size.


In [3]:
# CELL 2-A: Define the 30-document industrial knowledge base

KNOWLEDGE_BASE = [
    {"id": "SOP-MECH-001", "category": "SOP", "title": "Lockout/Tagout (LOTO) Procedure",
     "text": ("SOP-MECH-001: Lockout/Tagout (LOTO) Procedure. Purpose: Prevent unexpected "
              "energization of equipment during maintenance. Scope: All mechanical, electrical, "
              "and pneumatic equipment in the facility. Steps: (1) Notify control room and "
              "obtain work permit. (2) Identify all energy sources: electrical, pneumatic, "
              "hydraulic, gravity. (3) Shut down equipment using normal stopping procedure. "
              "(4) Isolate each energy source: close isolation valves, open electrical "
              "disconnects. (5) Apply LOTO lock and tag at each isolation point. Each "
              "technician applies their own personal lock. (6) Drain or bleed residual "
              "energy: release pressure, drain fluids, block gravity loads. (7) Verify zero "
              "energy state using appropriate test equipment. (8) Proceed with maintenance "
              "work. Restoration: Remove locks in reverse order, verify all personnel clear, "
              "restore energy, notify control room. Documentation: LOTO log must be signed "
              "before and after work. Responsibility: Authorized technician + safety officer.")},
    {"id": "SOP-SAFE-002", "category": "SOP", "title": "Hot Work Permit Procedure",
     "text": ("SOP-SAFE-002: Hot Work Permit Procedure. Purpose: Control fire and explosion "
              "risks during welding, cutting, grinding, or any spark-generating activity. "
              "Scope: All hot work activities in production, warehouse, and utility areas. "
              "Pre-work requirements: (1) Identify combustible materials within 10-meter "
              "radius and remove or shield them. (2) Check LEL (Lower Explosive Limit) - "
              "must be below 10% before work starts. (3) Position fire extinguisher "
              "(minimum 6kg CO2) within arm's reach. (4) Assign fire watch - dedicated "
              "person monitoring during and 30 minutes after work. (5) Obtain hot work "
              "permit signed by area supervisor and HSE officer. PPE required: welding "
              "helmet, flame-resistant gloves, leather apron, safety boots. Prohibited "
              "locations: within 5 meters of fuel storage, in confined spaces without "
              "ventilation assessment. Validity: 8 hours per permit.")},
    {"id": "SOP-SAFE-003", "category": "SOP", "title": "Confined Space Entry Procedure",
     "text": ("SOP-SAFE-003: Confined Space Entry Procedure. Confined spaces include tanks, "
              "vessels, silos, pits, and any enclosed space not designed for continuous "
              "occupancy. Entry requirements: (1) Obtain confined space entry permit signed "
              "by area supervisor. (2) Test atmosphere: O2 must be 19.5-23.5%, LEL below 10%, "
              "toxic gases below PEL. (3) Continuous ventilation using forced-air blower. "
              "(4) Assign entry supervisor, attendant (outside), and entrants. "
              "(5) Establish communication protocol every 5 minutes. "
              "(6) Retrieve system: tripod + lifeline for vertical entry. Emergency rescue: "
              "do NOT enter to rescue without own SCBA. Call emergency response team. "
              "Prohibited: entry without valid permit, entry without atmospheric test, "
              "entry without attendant present.")},
    {"id": "SOP-SAFE-004", "category": "SOP", "title": "Chemical Spill Response Procedure",
     "text": ("SOP-SAFE-004: Chemical Spill Response Procedure. Immediate actions upon spill: "
              "(1) Alert all personnel in area - evacuate if unknown chemical. (2) Identify "
              "chemical from SDS (Safety Data Sheet). (3) Assess spill size: minor (<5 liters), "
              "moderate (5-50 liters), major (>50 liters). Minor spill: trained personnel with "
              "PPE may clean up using spill kit. Moderate spill: notify HSE and supervisor. "
              "Do not enter without full PPE. Contain with absorbent booms. Major spill: "
              "evacuate 30-meter radius, notify emergency response team. Disposal: all spill "
              "waste in labeled hazardous waste containers. Report: all spills documented "
              "in incident report within 2 hours.")},
    {"id": "SOP-SAFE-005", "category": "SOP", "title": "PPE Requirements",
     "text": ("SOP-SAFE-005: Personal Protective Equipment Requirements by Work Zone. "
              "Zone A (General Plant): Safety helmet, safety shoes (steel toe), high-vis vest, "
              "safety glasses. Zone B (Chemical Handling): Zone A + chemical-resistant gloves, "
              "face shield, chemical apron. Zone C (Electrical Work): Zone A + arc-flash suit "
              "(8 cal/cm2 minimum), rubber-insulated gloves (Class 2), face shield with arc "
              "rating. Zone D (Welding/Hot Work): Safety helmet with welding shield "
              "(auto-darkening minimum shade 10), flame-resistant gloves, leather apron. "
              "Zone E (Confined Space): Zone A + harness, lifeline, gas detector. "
              "PPE inspection: visual check before each use. Damaged PPE must be removed "
              "from service immediately. Training required before entering any work zone.")},
    {"id": "SOP-SAFE-006", "category": "SOP", "title": "Work at Heights Procedure",
     "text": ("SOP-SAFE-006: Work at Heights Procedure. Definition: any work where a person "
              "could fall 1.8 meters or more. Equipment: Full-body harness, double-lanyard "
              "with energy absorber, anchor point rated minimum 22kN. Ladders: inspect before "
              "use, maintain 3-point contact, never stand on top two rungs, secure top and "
              "bottom. Scaffolding: inspect by certified inspector before use, tag-out any "
              "defective scaffold. Prohibited: working on unsecured ladders in wind >45 km/h, "
              "using scaffold without handrail, using harness with expired inspection date. "
              "Dropped object prevention: tool lanyards for all tools above 2 meters, "
              "toe boards on scaffold platforms, barricade below work area.")},
    {"id": "SOP-MECH-007", "category": "SOP", "title": "Machine Guarding Requirements",
     "text": ("SOP-MECH-007: Machine Guarding Requirements. Purpose: Prevent contact with "
              "moving parts - rotating shafts, gears, belts, pulleys, and cutting edges. "
              "Guard types: (1) Fixed guards - permanent barriers, must withstand 115 kg impact. "
              "(2) Interlocked guards - safety switch stops machine when guard opened. "
              "(3) Adjustable guards - for variable-feed machines. Inspection: monthly by "
              "maintenance, logged in equipment register. Guard removal requires work permit "
              "and machine lockout. Temporary removal allowed maximum 4 hours. "
              "Damaged or missing guards must be reported immediately. Machine tagged out "
              "until guard restored.")},
    {"id": "SOP-ELEC-008", "category": "SOP", "title": "Electrical Isolation Procedure",
     "text": ("SOP-ELEC-008: Electrical Isolation Procedure. Scope: All work on or near "
              "energized equipment above 50V. Qualified persons only - must hold valid "
              "electrical authorization card. Steps: (1) Identify all power sources - "
              "check single-line diagram. (2) De-energize at highest-level switch first. "
              "(3) Apply LOTO lock at each isolation point. (4) Verify de-energization "
              "using calibrated voltage tester - test all phases L1/L2/L3, L-N, L-PE. "
              "(5) Apply grounds on high-voltage systems (>1kV). "
              "Minimum PPE: Arc flash suit rated for incident energy. "
              "Prohibited: assuming de-energized without testing, bypassing interlocks, "
              "working alone on HV systems.")},
    {"id": "EQUIP-P101", "category": "Equipment Manual", "title": "Centrifugal Pump P-101",
     "text": ("EQUIP-P101: Centrifugal Pump P-101 Operations Manual. Type: horizontal "
              "end-suction centrifugal pump. Capacity: 120 m3/h. Head: 45 meters. "
              "Speed: 1450 RPM. Motor: 22 kW, 380V, 3-phase. Fluid: cooling water, "
              "max 60 deg C. Bearings: SKF 6309 (drive end), SKF 6307 (non-drive end). "
              "Lubrication: grease every 2000 operating hours. Seal: Flowserve 2100. "
              "Startup: (1) Open suction valve fully. (2) Prime pump. (3) Start motor. "
              "(4) Slowly open discharge valve. (5) Check vibration and noise. "
              "Normal vibration: <4.5 mm/s. Vibration alarm: >7.1 mm/s, shutdown: >11.2 mm/s. "
              "Bearing temp alarm: >85 deg C. Maintenance: bearing inspection 6 months, "
              "seal inspection 12 months, impeller inspection 24 months.")},
    {"id": "EQUIP-K202", "category": "Equipment Manual", "title": "Air Compressor K-202",
     "text": ("EQUIP-K202: Air Compressor K-202 Technical Manual. Type: rotary screw, "
              "oil-injected. Capacity: 500 Nm3/h FAD. Pressure: 8 bar g. Motor: 75 kW. "
              "Oil: ISO VG 46, change every 4000 hours. Air filter change: 2000 hours. "
              "High-temperature shutdown: 110 deg C. Low oil pressure shutdown: 2.5 bar. "
              "Overheating causes: blocked oil cooler, failed thermostatic valve, low oil "
              "level, ambient >40 deg C. Troubleshooting K-202 shutdown: check fault panel, "
              "acknowledge alarm, verify oil level, check cooler, check ambient temperature. "
              "Do not restart more than 3 times without investigating root cause.")},
    {"id": "EQUIP-HE302", "category": "Equipment Manual", "title": "Heat Exchanger HE-302",
     "text": ("EQUIP-HE302: Shell & Tube Heat Exchanger HE-302. Shell side: cooling water "
              "inlet 30 deg C, outlet 42 deg C, flow 80 m3/h. Tube side: process fluid "
              "inlet 75 deg C, outlet 50 deg C, flow 40 m3/h. Material: carbon steel shell, "
              "SS316L tubes. Fouling symptoms: increased pressure drop, reduced heat duty, "
              "outlet temperature deviation >5 deg C from design. Cleaning: chemical cleaning "
              "with 2% citric acid solution circulated 4 hours at 50 deg C. Mechanical tube "
              "cleaning for severe fouling. U-value monitoring monthly. U-value below 60% of "
              "design indicates cleaning needed.")},
    {"id": "EQUIP-GB103", "category": "Equipment Manual", "title": "Gearbox GB-103",
     "text": ("EQUIP-GB103: Gearbox GB-103. Type: helical gear reducer. Ratio: 4.5:1. "
              "Oil: ISO VG 220, volume 3.2 liters. Oil change: 500 hours first change, "
              "then 4000 hours or annually. Abnormal oil: viscosity change >15%, water >0.1%, "
              "Fe >150 ppm - investigate immediately. Acceptable vibration: <4 mm/s RMS. "
              "Abnormal noise: gear wear (whining), bearing wear (rumbling), misalignment "
              "(knocking). Shaft alignment: max angular 0.05mm/100mm, max parallel 0.1mm. "
              "Check alignment after any motor or gearbox removal.")},
    {"id": "EQUIP-M305", "category": "Equipment Manual", "title": "Electric Motor M-305",
     "text": ("EQUIP-M305: Electric Motor M-305. Rating: 15 kW, 380V, 3-phase, 1460 RPM. "
              "Insulation class F. Protection IP55. Bearings: 6309 (DE), 6307 (NDE). "
              "Lubrication: SKF LGWA2 grease, 15g per bearing every 3000 hours. "
              "DO NOT over-grease - excess grease overheats bearings. "
              "Megger test: L1/L2/L3 to earth at 500V DC. Acceptable: >100 MOhm (new), "
              ">10 MOhm (in-service), >1 MOhm (minimum). Below 1 MOhm: do not energize. "
              "Motor fails to start: (1) Check power supply, (2) Verify contactor, "
              "(3) Check megger, (4) Check mechanical binding, (5) Check overload relay.")},
    {"id": "EQUIP-CT101", "category": "Equipment Manual", "title": "Cooling Tower CT-101",
     "text": ("EQUIP-CT101: Cooling Tower CT-101. Type: induced-draft counterflow. "
              "Capacity: 500 RT. Inlet water 42 deg C, outlet 32 deg C. Fan: 132 kW, 8 blades. "
              "Water treatment: pH 7.0-7.5, conductivity <1500 uS/cm, biocide 2x weekly. "
              "Fan blade inspection monthly - check for cracks, erosion, imbalance. "
              "Replace blade if any crack or tip erosion >10mm. Vibration >5 mm/s requires "
              "dynamic balancing. Basin cleaning every 6 months. Legionella test every 3 months. "
              "Winter: maintain minimum flow, basin heater below 5 deg C ambient.")},
    {"id": "EQUIP-HV201", "category": "Equipment Manual", "title": "Control Valve HV-201",
     "text": ("EQUIP-HV201: Control Valve HV-201. Type: globe valve, pneumatic actuator, "
              "fail-close. Actuator: 3-15 PSI signal, 40-80 PSI supply. Positioner: Fisher 3582, "
              "4-20 mA input. Troubleshooting - valve not responding: (1) Check air supply "
              "(should be 80 PSI), (2) Check positioner signal (4-20 mA), "
              "(3) Manually stroke via handwheel, (4) Check body for buildup, "
              "(5) Check packing tightness. Packing replacement: every 2 years. "
              "Full stroke test monthly from control room, response time <15 seconds.")},
    {"id": "EQUIP-BC101", "category": "Equipment Manual", "title": "Belt Conveyor BC-101",
     "text": ("EQUIP-BC101: Belt Conveyor BC-101. Length: 45m, Width: 650mm, Speed: 1.2 m/s, "
              "Capacity: 80 tonnes/hour. Belt tension: max deflection 20mm under 10kg at midspan. "
              "Tracking: belt must run centered +/- 20mm. Belt alignment: start empty, observe "
              "tracking, adjust tail pulley - move toward drift side, max 1/4 turn per adjustment. "
              "Belt slip: indicates worn lagging - replace when slip >5%. Idler inspection monthly. "
              "Emergency stops: rope-pull every 15 meters, test monthly - stop within 2 seconds. "
              "Belt scraper inspection weekly.")},
    {"id": "SAP-GUIDE-001", "category": "SAP/ERP Guide", "title": "ME21N Create Purchase Order",
     "text": ("SAP-GUIDE-001: Transaction ME21N - Create Purchase Order. "
              "Required: Vendor number, Purchasing Organization, Material, Quantity, Delivery date. "
              "Error M8 147 - vendor master not approved for company code: contact procurement "
              "to activate vendor in FK01. Error ME 073 - material master missing purchasing view. "
              "Error ME 013 - purchasing info record not found, create with ME11. "
              "Approval: PO above IDR 50M requires supervisor, above 500M requires manager. "
              "After save: PO number format 45XXXXXXXX. GR against PO: use MIGO movement type 101.")},
    {"id": "SAP-GUIDE-002", "category": "SAP/ERP Guide", "title": "MIGO Goods Receipt",
     "text": ("SAP-GUIDE-002: Transaction MIGO - Goods Receipt Movement Type 101. "
              "Select Goods Receipt + Purchase Order, enter PO number, click Execute. "
              "For each line: verify quantity, enter storage location, check Item OK. "
              "Post to generate Material Document. "
              "Error M7 021 - quantity exceeds PO quantity: amend PO. "
              "Error M7 053 - storage location not defined: contact MM team. "
              "PO closed status: re-open via ME22N, remove Delivery Completed indicator. "
              "Cancel GR: MIGO Cancellation + Material Document.")},
    {"id": "SAP-GUIDE-003", "category": "SAP/ERP Guide", "title": "MIRO Invoice Verification",
     "text": ("SAP-GUIDE-003: Transaction MIRO - Logistics Invoice Verification. "
              "Purpose: 3-way match: invoice vs PO vs GR. "
              "Enter invoice date, reference, currency, company code, then PO number. "
              "Amount mismatch error F5 703: (1) Contact vendor for credit/debit note, "
              "(2) Amend PO price in ME22N, (3) Finance posts difference to expense account. "
              "Payment block: set indicator R for disputed invoices. "
              "After posting: document number format 51XXXXXXXX. "
              "Report pending invoices: MIR5.")},
    {"id": "SAP-GUIDE-004", "category": "SAP/ERP Guide", "title": "IW51 Maintenance Notification",
     "text": ("SAP-GUIDE-004: Transaction IW51 - Create Maintenance Notification. "
              "Types: M2 malfunction report, M1 maintenance request. "
              "Mandatory: Notification type, Short text (40 chars), Functional location or Equipment. "
              "Error IW 038 - mandatory field empty. "
              "After creation: notification number format 10XXXXXXXX. "
              "Status flow: OSNO (outstanding) > INPR (in process) > NOCO (completed). "
              "SLA: Priority 1 = 4 hours, Priority 2 = 24 hours. "
              "Mobile entry: Fiori app Create Maintenance Notification.")},
    {"id": "SAP-GUIDE-005", "category": "SAP/ERP Guide", "title": "CO01 Production Order",
     "text": ("SAP-GUIDE-005: Transaction CO01 - Create Production Order. "
              "Inputs: Material, Plant, Production version, Order type PP01, Quantity, Dates. "
              "Error CO 280 - BOM not active: activate in CS01/CS02. "
              "Error CO 417 - routing not found: create in CA01. "
              "Release: CO02 > Release triggers material availability check. "
              "Goods issue: MIGO movement type 261. Goods receipt: MIGO 101 with order. "
              "Confirmation: CO11N - report quantities, machine time, labor.")},
    {"id": "SAP-GUIDE-006", "category": "SAP/ERP Guide", "title": "MB51 Material Documents",
     "text": ("SAP-GUIDE-006: Transaction MB51 - Material Document List. "
              "Movement types: 101 (GR from PO), 201 (GI to cost center), "
              "261 (GI to production order), 301 (plant transfer), 501 (GR without PO). "
              "Batch job failure: check SM37 job log. Causes: lock conflict, memory dump, "
              "variant not saved. Fix: re-trigger in SM36, reduce query scope, recreate variant. "
              "Export to Excel: List > Save/Send > File > Spreadsheet.")},
    {"id": "SAP-GUIDE-007", "category": "SAP/ERP Guide", "title": "SAP User Role Management",
     "text": ("SAP-GUIDE-007: SAP User Role Management. "
              "User creation: SU01. Role assignment: SU01 > Roles tab. "
              "User cannot access transaction: check SU53 for missing authorization. "
              "Verify role in SU01 is saved and not locked. User may need to re-login. "
              "Access request: user submits form > business owner approves > IT assigns > verify. "
              "Password reset: SU01 > Logon Data. Locked accounts: SU01 > unlock. "
              "Auto-lock after 3 failed attempts. Access review every 6 months.")},
    {"id": "IT-RB-001", "category": "IT/Network Runbook", "title": "SCADA Connectivity",
     "text": ("IT-RB-001: SCADA Connectivity Troubleshooting. "
              "Symptoms: cannot communicate with PLC, tags showing stale quality, timeouts. "
              "(1) Ping PLC IP from SCADA server. (2) Verify NIC settings - IP in same "
              "subnet as PLC (192.168.10.x/24). (3) Check VLAN assignment (should be VLAN 10). "
              "(4) Check PLC RUN LED - must be solid green. "
              "(5) Verify Kepware OPC Server service Running. "
              "(6) Firewall: port 102 (Siemens S7), 502 (Modbus TCP), 44818 (EtherNet/IP). "
              "Escalate with switch port capture and SCADA event log. "
              "Recovery estimate: network 30min, PLC 2hr, SCADA reinstall 4hr.")},
    {"id": "IT-RB-002", "category": "IT/Network Runbook", "title": "VPN Reset Procedure",
     "text": ("IT-RB-002: VPN Reset Procedure. Platform: Cisco AnyConnect to HQ. "
              "User reset: (1) Disconnect AnyConnect. (2) Clear profile folder under "
              "ProgramData Cisco AnyConnect. (3) Reconnect with fresh credentials. "
              "VPN drops every 2 hours: session timeout issue. Admin: increase Session Timeout "
              "from 7200s to 28800s in VPN concentrator Group Policy. "
              "Certificate error: check system clock is NTP-synced. "
              "Check CA certificate in Windows trusted root store. "
              "Max 500 simultaneous sessions. Log: CiscoAnyConnect.log in Temp folder.")},
    {"id": "IT-RB-003", "category": "IT/Network Runbook", "title": "Backup Storage Cleanup",
     "text": ("IT-RB-003: Database Backup Storage Cleanup. "
              "Trigger: storage >85% full, backup job failing. "
              "Retention: full backups - keep last 4 weekly + 12 monthly. "
              "Transaction logs - keep last 7 days. "
              "Safe to delete: full backups older than 3 months not tagged monthly-keep. "
              "Transaction logs older than 7 days. Temp files. "
              "Do NOT delete: tagged monthly-keep or audit-retain. Quarter-end files. "
              "PowerShell cleanup: Get-ChildItem -Recurse | Where LastWriteTime older "
              "than 7 days and Name like *.trn | Remove-Item. "
              "Alert threshold: configure at 80% for proactive cleanup.")},
    {"id": "IT-RB-004", "category": "IT/Network Runbook", "title": "NTP Synchronization",
     "text": ("IT-RB-004: NTP Time Synchronization Procedure. "
              "Hierarchy: GPS (Stratum 1) > primary NTP server > Windows DC > clients. "
              "Check status: w32tm /query /status. Force resync: w32tm /resync /force. "
              "If fails: net stop w32tm then net start w32tm, then w32tm /resync. "
              "PLC time sync (Siemens S7): configure in Step7 NetPro with SCADA server IP. "
              "Maximum acceptable drift for SAP-PLC consistency: +/- 2 minutes. "
              "Current issue: PLC timestamp differs from SAP by 15 minutes - causes "
              "journal entry date mismatches in SAP PM. "
              "Resolution: force NTP resync, update PLC time via Step7.")},
    {"id": "IT-RB-005", "category": "IT/Network Runbook", "title": "Firewall Whitelist",
     "text": ("IT-RB-005: Firewall Whitelist - Add New IP/Port Exception. "
              "Platform: Palo Alto PA-3220, managed via Panorama. "
              "Process: (1) Submit request with source/dest IP, port, justification, approver. "
              "(2) Network team reviews - minimum 1 business day. "
              "(3) Approved in ServiceNow CHG ticket. "
              "(4) Admin implements in Panorama > Policies > Security. "
              "Rule naming: TICKET-SRCZONE-DSTZONE-DATE. "
              "Supplier portal access: whitelist outbound to supplier FQDN port 443. "
              "Rules inactive 90 days flagged for removal. Quarterly audit required.")},
    {"id": "IT-RB-006", "category": "IT/Network Runbook", "title": "Printer Driver Rollback",
     "text": ("IT-RB-006: Printer Driver Rollback Procedure. "
              "Trigger: printer offline after Windows Update or firmware update. "
              "Step 1: Settings > Devices > Printers. If Offline: uncheck Use Printer Offline. "
              "Step 2: Remove driver - Print Server Properties > Drivers > "
              "Remove Driver and Driver Package. "
              "Step 3: Install previous driver from IT shared drive previous_version folder. "
              "Step 4: Re-add printer by IP address. Step 5: Test print. "
              "If fails: check firewall - port 9100 and 515 required. "
              "Prevention: Group Policy disable automatic driver updates.")},
    {"id": "IT-RB-007", "category": "IT/Network Runbook", "title": "PI Historian Resync",
     "text": ("IT-RB-007: OSIsoft PI Historian Resync / Data Gap Recovery. "
              "Symptoms: PI tags showing No Data, data gap in PI Vision trending. "
              "Causes: PI Interface stopped, network disconnection, PI Buffer overflow. "
              "Recovery: (1) Start PI Interface Manager - verify all interfaces Running. "
              "(2) Check PIBUFSS service running. (3) Buffered data auto-replays when "
              "connection restored - monitor in PI SMT > Operation. "
              "(4) If buffer overflow: export CSV from DCS, import via PI DataLink or WebAPI. "
              "Data gap >6 hours requires incident report. "
              "PI Buffer configured for 72-hour capacity. Alert at 80%. "
              "Validate: verify PI values match DCS at 3 random timestamps.")},
]

import pandas as pd
df_kb = pd.DataFrame([
    {'id': d['id'], 'category': d['category'], 'title': d['title'],
     'text_len': len(d['text'].split())}
    for d in KNOWLEDGE_BASE
])
print(f"Knowledge base: {len(KNOWLEDGE_BASE)} documents")
print()
print(df_kb.groupby('category').agg(count=('id','count'), avg_words=('text_len','mean')).round(0).to_string())
print(f"Total words: {df_kb['text_len'].sum()}")


Knowledge base: 30 documents

                    count  avg_words
category                            
Equipment Manual        8       83.0
IT/Network Runbook      7       84.0
SAP/ERP Guide           7       70.0
SOP                     8      104.0
Total words: 2579


In [4]:
# CELL 2-B: Document chunking - fixed-size and sentence-aware

def chunk_fixed(text, doc_id, chunk_size=200, overlap=40):
    '''Fixed-size word chunking with overlap.''' 
    words = text.split()
    chunks, start, chunk_idx = [], 0, 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append({
            'chunk_id': f'{doc_id}::chunk{chunk_idx:02d}',
            'doc_id': doc_id, 'method': 'fixed',
            'text': ' '.join(words[start:end]),
        })
        if end == len(words):
            break
        start += chunk_size - overlap
        chunk_idx += 1
    return chunks


def chunk_sentences(text, doc_id, target_chars=500, overlap_sentences=1):
    '''Sentence-aware chunking.''' 
    import re
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    sents = [s.strip() for s in sents if s.strip()]
    chunks, cur, cur_len, idx = [], [], 0, 0
    for s in sents:
        cur.append(s)
        cur_len += len(s)
        if cur_len >= target_chars:
            chunks.append({'chunk_id': f'{doc_id}::sent{idx:02d}',
                           'doc_id': doc_id, 'method': 'sentence',
                           'text': ' '.join(cur)})
            cur = cur[-overlap_sentences:] if overlap_sentences else []
            cur_len = sum(len(x) for x in cur)
            idx += 1
    if cur:
        chunks.append({'chunk_id': f'{doc_id}::sent{idx:02d}',
                       'doc_id': doc_id, 'method': 'sentence',
                       'text': ' '.join(cur)})
    return chunks


all_fixed, all_sent = [], []
for doc in KNOWLEDGE_BASE:
    all_fixed.extend(chunk_fixed(doc['text'], doc['id']))
    all_sent.extend(chunk_sentences(doc['text'], doc['id']))

chunks = all_sent  # use sentence chunks as primary

print(f"Fixed-size chunking:     {len(all_fixed)} chunks")
print(f"Sentence-aware chunking: {len(all_sent)} chunks")
lengths = [len(c['text'].split()) for c in chunks]
print(f"Sentence chunk stats: min={min(lengths)}, max={max(lengths)}, "
      f"mean={sum(lengths)/len(lengths):.0f}, median={sorted(lengths)[len(lengths)//2]}")
sample = next(c for c in chunks if 'SOP-MECH-001' in c['chunk_id'])
print(f"Sample chunk (SOP-MECH-001::sent00):  {sample['text'][:200]}...")


Fixed-size chunking:     30 chunks
Sentence-aware chunking: 53 chunks
Sentence chunk stats: min=3, max=87, mean=53, median=69
Sample chunk (SOP-MECH-001::sent00):  SOP-MECH-001: Lockout/Tagout (LOTO) Procedure. Purpose: Prevent unexpected energization of equipment during maintenance. Scope: All mechanical, electrical, and pneumatic equipment in the facility. Ste...


---
## Section 3 — Embeddings: Turning Text into Vectors

An embedding is a dense numerical vector representing the *semantic meaning* of text.

```
"LOTO tidak dilakukan"    →  [0.23, -0.87, ..., 0.56]  ← 384 numbers
"Lockout procedure skipped" →  [0.21, -0.85, ..., 0.54]  ← similar vectors!
```

This differs from TF-IDF (Day 1): TF-IDF sees "LOTO" and "lockout" as completely different
(0 overlap), while embeddings capture their shared meaning (high similarity).

### Model: paraphrase-multilingual-MiniLM-L12-v2

| Property | Value |
|---|---|
| Embedding dimension | 384 |
| Languages | 50+ (including Indonesian) |
| Model size | ~118 MB |
| Speed | ~1500 sentences/second (CPU) |

### Cosine similarity

$$\text{cosine}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$

Range: -1 to +1. Values above 0.7 indicate strong semantic similarity.


In [5]:
# CELL 3-A: Encode all chunks with sentence-transformers (or mock)

import numpy as np

def get_embeddings(texts):
    if EMBED_PROVIDER == 'sentence_transformers':
        return EMBED_MODEL.encode(texts, show_progress_bar=True, normalize_embeddings=True)
    import hashlib
    vecs = []
    for text in texts:
        seed = int(hashlib.md5(text.encode()).hexdigest(), 16) % (2**32)
        rng = np.random.default_rng(seed)
        v = rng.standard_normal(384).astype(np.float32)
        vecs.append(v / np.linalg.norm(v))
    return np.array(vecs)


chunk_texts = [c['text'] for c in chunks]
chunk_ids   = [c['chunk_id'] for c in chunks]

print(f"Encoding {len(chunk_texts)} chunks...")
t0 = time.time()
chunk_embeddings = get_embeddings(chunk_texts)
elapsed = time.time() - t0

print(f"✓ Done in {elapsed:.1f}s")
print(f"Matrix shape: {chunk_embeddings.shape}  ({chunk_embeddings.shape[0]} chunks x {chunk_embeddings.shape[1]} dims)")
print(f"dtype: {chunk_embeddings.dtype},  L2 norm[0]: {np.linalg.norm(chunk_embeddings[0]):.4f} (should be ~1.0)")

if EMBED_PROVIDER == 'mock':
    print("⚠  MOCK embeddings (random, normalized). Retrieval quality is not semantic.")
    print("   Install sentence-transformers for real multilingual semantic search.")
else:
    print("✓ Real multilingual embeddings loaded.")


Encoding 53 chunks...


Batches: 100%|██████████| 2/2 [00:00<00:00, 14.69it/s]

✓ Done in 0.1s
Matrix shape: (53, 384)  (53 chunks x 384 dims)
dtype: float32,  L2 norm[0]: 1.0000 (should be ~1.0)
✓ Real multilingual embeddings loaded.


In [7]:
# CELL 3-B: Cosine similarity and top-k retrieval

def cosine_sim(query_vec, matrix):
    return np.dot(matrix, query_vec)  # works when both are unit-normalized


def retrieve_top_k(query_text, k=3, chunk_vecs=None, chunk_list=None):
    if chunk_vecs is None: chunk_vecs = chunk_embeddings
    if chunk_list is None: chunk_list = chunks

    if EMBED_PROVIDER == 'mock':
        # Keyword routing for pedagogical consistency in mock mode
        q = query_text.lower()
        if any(w in q for w in ['loto', 'safety', 'apd', 'permit', 'spill', 'hot work']):
            cands = [c for c in chunk_list if any(x in c['doc_id'] for x in ['SOP-SAFE','SOP-MECH','SOP-ELEC'])]
        elif any(w in q for w in ['me21n','migo','miro','iw51','co01','sap','purchase','mb51']):
            cands = [c for c in chunk_list if 'SAP-GUIDE' in c['doc_id']]
        elif any(w in q for w in ['scada','vpn','network','printer','ntp','firewall','historian','pi ']):
            cands = [c for c in chunk_list if 'IT-RB' in c['doc_id']]
        elif any(w in q for w in ['pump','pompa','p-101','compressor','bearing','valve','motor','conveyor']):
            cands = [c for c in chunk_list if 'EQUIP' in c['doc_id']]
        else:
            cands = chunk_list[:k*3]
        return [(c, 0.85 - i*0.05) for i, c in enumerate(cands[:k])]

    qv = get_embeddings([query_text])[0]
    sims = cosine_sim(qv, chunk_vecs)
    idxs = np.argsort(sims)[::-1][:k]
    return [(chunk_list[i], float(sims[i])) for i in idxs]


DEMO_QUERIES = [
    "LOTO procedure for pump maintenance",
    "ME21N purchase order error vendor master not approved",
    "SCADA server cannot connect to PLC network timeout",
]

for query in DEMO_QUERIES:
    print("=" * 60)
    print(f"Query: {query}")
    print("-" * 60)
    for rank, (chunk, score) in enumerate(retrieve_top_k(query, k=3), 1):
        print(f"  Rank {rank} | Score: {score:.3f} | {chunk['chunk_id']}")
        print(f"    {chunk['text'][:110]}...")
    print()


Query: LOTO procedure for pump maintenance
------------------------------------------------------------


Batches: 100%|██████████| 1/1 [00:00<00:00, 128.75it/s]


  Rank 1 | Score: 0.593 | EQUIP-P101::sent00
    EQUIP-P101: Centrifugal Pump P-101 Operations Manual. Type: horizontal end-suction centrifugal pump. Capacity:...
  Rank 2 | Score: 0.508 | SOP-MECH-001::sent00
    SOP-MECH-001: Lockout/Tagout (LOTO) Procedure. Purpose: Prevent unexpected energization of equipment during ma...
  Rank 3 | Score: 0.478 | SOP-MECH-001::sent01
    (5) Apply LOTO lock and tag at each isolation point. Each technician applies their own personal lock. (6) Drai...

Query: ME21N purchase order error vendor master not approved
------------------------------------------------------------


Batches: 100%|██████████| 1/1 [00:00<00:00, 323.11it/s]


  Rank 1 | Score: 0.655 | SAP-GUIDE-001::sent00
    SAP-GUIDE-001: Transaction ME21N - Create Purchase Order. Required: Vendor number, Purchasing Organization, Ma...
  Rank 2 | Score: 0.393 | SAP-GUIDE-005::sent00
    SAP-GUIDE-005: Transaction CO01 - Create Production Order. Inputs: Material, Plant, Production version, Order ...
  Rank 3 | Score: 0.365 | SOP-MECH-001::sent02
    Responsibility: Authorized technician + safety officer....

Query: SCADA server cannot connect to PLC network timeout
------------------------------------------------------------


Batches: 100%|██████████| 1/1 [00:00<00:00, 342.76it/s]

  Rank 1 | Score: 0.675 | IT-RB-001::sent00
    IT-RB-001: SCADA Connectivity Troubleshooting. Symptoms: cannot communicate with PLC, tags showing stale quali...
  Rank 2 | Score: 0.506 | IT-RB-001::sent01
    Recovery estimate: network 30min, PLC 2hr, SCADA reinstall 4hr....
  Rank 3 | Score: 0.490 | IT-RB-004::sent00
    IT-RB-004: NTP Time Synchronization Procedure. Hierarchy: GPS (Stratum 1) > primary NTP server > Windows DC > ...



---
## Section 4 — Vector Databases: FAISS & ChromaDB

| Feature | FAISS | ChromaDB | Milvus | Qdrant |
|---|---|---|---|---|
| **Type** | Library | Embedded/Server | Distributed | Server |
| **Persistence** | Save to disk | SQLite/DuckDB | External | On-disk |
| **Metadata filter** | No | Yes | Yes | Yes |
| **Best for** | Research | Local app, Jupyter | Production scale | Production |
| **Install** | `faiss-cpu` | `chromadb` | `pymilvus` | `qdrant-client` |

### Index types
- **Flat** (IndexFlatIP): exact search, 100% recall, linear time
- **IVF**: partition into clusters, search nearby clusters only  
- **HNSW**: graph-based ANN, excellent recall/speed tradeoff

**Tonight:** FAISS `IndexFlatIP` (exact) + ChromaDB (persistent, metadata).


In [8]:
# CELL 4-A: Build FAISS IndexFlatIP

class MockFAISS:
    '''Pure-numpy FAISS substitute for offline use.''' 
    def __init__(self, dim):
        self.dim = dim
        self.vectors = None
    def add(self, vecs):
        self.vectors = vecs.copy() if self.vectors is None else np.vstack([self.vectors, vecs])
    def search(self, qv, k):
        sims = np.dot(qv, self.vectors.T)
        top = np.argsort(sims, axis=1)[:, ::-1][:, :k]
        return np.take_along_axis(sims, top, axis=1), top


try:
    import faiss
    dim = chunk_embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dim)
    faiss_index.add(chunk_embeddings.astype(np.float32))
    print(f"✓ FAISS IndexFlatIP: {faiss_index.ntotal} vectors, dim={dim}")
except ImportError:
    faiss_index = MockFAISS(chunk_embeddings.shape[1])
    faiss_index.add(chunk_embeddings.astype(np.float32))
    print(f"⚠  faiss-cpu not installed → MockFAISS ({faiss_index.vectors.shape[0]} vectors)")
    print("   Install: pip install faiss-cpu")

QUERY_K05 = "Prosedur LOTO belum dilakukan sebelum teknisi masuk ke dalam tangki pembersihan"
qv = get_embeddings([QUERY_K05]).astype(np.float32)
scores, idxs = faiss_index.search(qv, k=3)

print(f"Query: {QUERY_K05}")
print(f"Top-3 FAISS results:")
print("-" * 60)
for rank, (idx, score) in enumerate(zip(idxs[0], scores[0]), 1):
    c = chunks[int(idx)]
    print(f"  Rank {rank} | Score: {score:.4f} | {c['chunk_id']}")
    print(f"    {c['text'][:150]}...")
    print()


✓ FAISS IndexFlatIP: 53 vectors, dim=384


Batches: 100%|██████████| 1/1 [00:00<00:00, 166.44it/s]

Query: Prosedur LOTO belum dilakukan sebelum teknisi masuk ke dalam tangki pembersihan
Top-3 FAISS results:
------------------------------------------------------------
  Rank 1 | Score: 0.4449 | SOP-MECH-001::sent01
    (5) Apply LOTO lock and tag at each isolation point. Each technician applies their own personal lock. (6) Drain or bleed residual energy: release pres...

  Rank 2 | Score: 0.3504 | SOP-MECH-001::sent00
    SOP-MECH-001: Lockout/Tagout (LOTO) Procedure. Purpose: Prevent unexpected energization of equipment during maintenance. Scope: All mechanical, electr...

  Rank 3 | Score: 0.3363 | EQUIP-HE302::sent00
    EQUIP-HE302: Shell & Tube Heat Exchanger HE-302. Shell side: cooling water inlet 30 deg C, outlet 42 deg C, flow 80 m3/h. Tube side: process fluid inl...



In [9]:
# CELL 4-B: Build ChromaDB collection

class MockChroma:
    def __init__(self, name):
        self.name = name
        self._ids, self._embs, self._docs, self._metas = [], [], [], []
    def add(self, ids, embeddings, documents, metadatas):
        self._ids.extend(ids); self._embs.extend(embeddings)
        self._docs.extend(documents); self._metas.extend(metadatas)
    def query(self, query_embeddings, n_results=3):
        q = np.array(query_embeddings[0])
        q /= (np.linalg.norm(q) + 1e-10)
        M = np.array(self._embs)
        M /= (np.linalg.norm(M, axis=1, keepdims=True) + 1e-10)
        sims = np.dot(M, q)
        top = np.argsort(sims)[::-1][:n_results]
        return {'ids': [[self._ids[i] for i in top]],
                'documents': [[self._docs[i] for i in top]],
                'metadatas': [[self._metas[i] for i in top]],
                'distances': [[1 - float(sims[i]) for i in top]]}
    @property
    def count(self): return len(self._ids)


try:
    import chromadb
    cc = chromadb.Client()
    collection = cc.create_collection(name='industrial_kb', metadata={'hnsw:space': 'cosine'})
    BATCH = 100
    for i in range(0, len(chunks), BATCH):
        bc = chunks[i:i+BATCH]; be = chunk_embeddings[i:i+BATCH]
        collection.add(ids=[c['chunk_id'] for c in bc], embeddings=be.tolist(),
                       documents=[c['text'] for c in bc],
                       metadatas=[{'doc_id': c['doc_id'], 'method': c['method']} for c in bc])
    print(f"✓ ChromaDB 'industrial_kb': {collection.count()} vectors")
except ImportError:
    collection = MockChroma('industrial_kb')
    for chunk, emb in zip(chunks, chunk_embeddings):
        collection.add(ids=[chunk['chunk_id']], embeddings=[emb.tolist()],
                       documents=[chunk['text']],
                       metadatas=[{'doc_id': chunk['doc_id'], 'method': chunk['method']}])
    print(f"⚠  chromadb not installed → MockChroma ({collection.count} vectors)")
    print("   Install: pip install chromadb")

results = collection.query(query_embeddings=[get_embeddings([QUERY_K05])[0].tolist()], n_results=3)

print(f"Query: {QUERY_K05}")
print("Top-3 ChromaDB results:")
for rank, (did, doc, meta, dist) in enumerate(zip(
        results['ids'][0], results['documents'][0],
        results['metadatas'][0], results['distances'][0]), 1):
    print(f"  Rank {rank} | Distance: {dist:.4f} | {did}")
    print(f"    doc_id: {meta['doc_id']}")
    print(f"    {doc[:150]}...")
    print()

print("→ FAISS and ChromaDB return the same logical results.")
print("  ChromaDB adds: metadata filtering, persistence, collection management.")


✓ ChromaDB 'industrial_kb': 53 vectors


Batches: 100%|██████████| 1/1 [00:00<00:00, 135.34it/s]

Query: Prosedur LOTO belum dilakukan sebelum teknisi masuk ke dalam tangki pembersihan
Top-3 ChromaDB results:
  Rank 1 | Distance: 0.5551 | SOP-MECH-001::sent01
    doc_id: SOP-MECH-001
    (5) Apply LOTO lock and tag at each isolation point. Each technician applies their own personal lock. (6) Drain or bleed residual energy: release pres...

  Rank 2 | Distance: 0.6496 | SOP-MECH-001::sent00
    doc_id: SOP-MECH-001
    SOP-MECH-001: Lockout/Tagout (LOTO) Procedure. Purpose: Prevent unexpected energization of equipment during maintenance. Scope: All mechanical, electr...

  Rank 3 | Distance: 0.6637 | EQUIP-HE302::sent00
    doc_id: EQUIP-HE302
    EQUIP-HE302: Shell & Tube Heat Exchanger HE-302. Shell side: cooling water inlet 30 deg C, outlet 42 deg C, flow 80 m3/h. Tube side: process fluid inl...

→ FAISS and ChromaDB return the same logical results.
  ChromaDB adds: metadata filtering, persistence, collection management.


---
## Section 5 — Context Injection & Prompt Assembly

### The RAG prompt template

```
SYSTEM: You are an industrial AI assistant. Answer ONLY using the context below.

CONTEXT:
[DOCUMENT 1: SOP-MECH-001]
{chunk_1_text}

[DOCUMENT 2: EQUIP-P101]
{chunk_2_text}

USER: {user_query}
```

### Token budget

| Component | Typical tokens |
|---|---|
| System prompt | 80–120 |
| 3 context chunks (200 words each) | ~850 |
| User query | 20–50 |
| **Total input** | **~1000** |
| LLM answer | 100–200 |

At Claude Haiku pricing ($0.25/M input): **$0.0003 per RAG query** — 1 cent per 33 queries.

### Citation format

Good RAG responses include citations:

> "According to SOP-MECH-001 (LOTO Procedure), step 5 requires a personal lock
> at each isolation point. [Source: SOP-MECH-001]"


In [13]:
# CELL 5-A: rag_query() - retrieve, format context, generate grounded answer

RAG_SYSTEM_PROMPT = (
    "You are an industrial AI assistant for a manufacturing facility. "
    "Answer questions ONLY using the context documents provided below. "
    "If the information is not in the context, say: I could not find this in the knowledge base. "
    "Always cite the source document ID, e.g. [Source: SOP-MECH-001], in your answer. "
    "Be concise and precise - this is a safety-critical environment."
)


def format_context(retrieved, max_chunks=3):
    parts = []
    for i, (chunk, score) in enumerate(retrieved[:max_chunks], 1):
        parts.append(f"[DOCUMENT {i}: {chunk['doc_id']}]{chunk['text']}")
    return "".join(parts)


def rag_query(user_query, k=3, verbose=False):
    '''Full RAG pipeline: retrieve -> format context -> generate grounded answer.''' 
    t0 = time.time()
    retrieved = retrieve_top_k(user_query, k=k)
    context_block = format_context(retrieved)
    user_message = f"Context:{context_block}Question: {user_query}"

    if verbose:
        print("=" * 60)
        print(f"QUERY: {user_query}")
        print(f"RETRIEVED ({len(retrieved)} chunks):")
        for c, s in retrieved:
            print(f"  {c['chunk_id']} (score={s:.3f})")
        print("-" * 60)

    answer = chat(
        messages=[{'role': 'user', 'content': user_message}],
        system=RAG_SYSTEM_PROMPT,
        _context_chunks=[c['text'] for c, _ in retrieved],
    )
    latency = time.time() - t0

    if verbose:
        print(f"ANSWER ({latency:.2f}s):{answer}")
        print("=" * 60)

    return answer, retrieved, latency


answer, retrieved, latency = rag_query(
    "What is the LOTO procedure for centrifugal pump P-101?",
    k=3, verbose=True,
)


Batches: 100%|██████████| 1/1 [00:00<00:00, 93.04it/s]


QUERY: What is the LOTO procedure for centrifugal pump P-101?
RETRIEVED (3 chunks):
  EQUIP-P101::sent00 (score=0.668)
  EQUIP-HV201::sent00 (score=0.420)
  EQUIP-HE302::sent00 (score=0.378)
------------------------------------------------------------
ANSWER (1.87s):I could not find this in the knowledge base.

The EQUIP-P101 document provides the operations manual for centrifugal pump P-101, including startup procedures, operating parameters, and vibration limits, but it does not contain a Lockout/Tagout (LOTO) procedure. [Source: EQUIP-P101]

For LOTO procedures, you would need to consult your facility's energy control program documentation or safety procedures manual.


In [14]:
# CELL 5-B: Side-by-side comparison - without RAG vs with RAG

TEST_QUERIES = [
    "What is the LOTO procedure for centrifugal pump P-101?",
    "ME21N purchase order fails - vendor master not approved",
    "SCADA server cannot connect to PLC, how do I troubleshoot?",
]

for query in TEST_QUERIES:
    print("=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)

    t0 = time.time()
    no_rag = chat(messages=[{'role': 'user', 'content': query}],
                  system='You are an industrial maintenance assistant.')
    lat_no = time.time() - t0

    rag_ans, retrieved, lat_rag = rag_query(query, k=3)

    print(f"Without RAG ({lat_no:.2f}s):")
    print(f"   {no_rag[:300]}...")

    print(f"With RAG ({lat_rag:.2f}s) - retrieved:")
    for c, s in retrieved:
        print(f"   [{s:.3f}] {c['chunk_id']}")
    print(f"   Answer: {rag_ans[:400]}...")
    print()

print("Key observations:")
print("  1. RAG answer cites specific document IDs (verifiable)")
print("  2. RAG answer uses exact terminology from the SOP/manual")
print("  3. No-RAG answer may contain invented details (hallucination)")
print("  4. RAG latency is slightly higher (retrieval adds ~50-200ms)")


QUERY: What is the LOTO procedure for centrifugal pump P-101?


Batches: 100%|██████████| 1/1 [00:00<00:00, 92.51it/s]


Without RAG (4.81s):
   # LOTO Procedure for Centrifugal Pump P-101

I don't have specific documentation for your P-101 pump, but I can provide a **standard LOTO (Lockout/Tagout) procedure** that should be adapted to your facility's requirements:

## General LOTO Steps:

### 1. **Preparation**
- Notify all affected personn...
With RAG (1.63s) - retrieved:
   [0.668] EQUIP-P101::sent00
   [0.420] EQUIP-HV201::sent00
   [0.378] EQUIP-HE302::sent00
   Answer: I could not find this in the knowledge base.

The EQUIP-P101 document provides the operations manual for centrifugal pump P-101, including startup procedures, operating parameters, and vibration limits, but it does not contain a Lockout/Tagout (LOTO) procedure. [Source: EQUIP-P101]

For LOTO procedures, you would need to consult your facility's energy control program documentation or safety proced...

QUERY: ME21N purchase order fails - vendor master not approved


Batches: 100%|██████████| 1/1 [00:00<00:00, 326.89it/s]


Without RAG (6.12s):
   # ME21N Purchase Order Error: Vendor Master Not Approved

## Problem
The purchase order creation in **ME21N** is failing because the vendor master record hasn't been approved/activated.

## Quick Solutions

### 1. **Check Vendor Status (FK02)**
```
Transaction: FK02 (Change Vendor Master)
→ Enter ve...
With RAG (1.98s) - retrieved:
   [0.677] SAP-GUIDE-001::sent00
   [0.418] SAP-GUIDE-005::sent00
   [0.307] SAP-GUIDE-002::sent00
   Answer: Based on the error you're experiencing:

**Error: M8 147 - Vendor master not approved for company code**

**Solution:** Contact procurement to activate the vendor in transaction **FK01** (Vendor Master).

The vendor number exists in SAP but is not approved for your specific company code. Procurement must authorize the vendor for your company code before you can create the purchase order in ME21N.
...

QUERY: SCADA server cannot connect to PLC, how do I troubleshoot?


Batches: 100%|██████████| 1/1 [00:00<00:00, 197.32it/s]


Without RAG (5.28s):
   # SCADA-to-PLC Connection Troubleshooting

## Quick Diagnostic Steps

### 1. **Verify Physical Connections**
- Check network cables (Ethernet) or serial connections for damage
- Ensure cables are fully seated in ports
- Look for bent pins or corrosion on connectors
- Test with a different cable if a...
With RAG (2.30s) - retrieved:
   [0.755] IT-RB-001::sent00
   [0.502] IT-RB-001::sent01
   [0.360] IT-RB-004::sent00
   Answer: Based on IT-RB-001, follow these troubleshooting steps for SCADA-to-PLC connectivity issues:

1. **Ping the PLC IP** from the SCADA server
2. **Verify NIC settings** - ensure SCADA server IP is in the same subnet as PLC (192.168.10.x/24)
3. **Check VLAN assignment** - should be VLAN 10
4. **Verify PLC RUN LED** - must be solid green
5. **Check Kepware OPC Server service** - confirm it's running
6....

Key observations:
  1. RAG answer cites specific document IDs (verifiable)
  2. RAG answer uses exact terminology from the SOP/manual
  3. 

---
## Section 6 — Full RAG Pipeline: All 40 Tickets

**Pipeline per ticket:**
1. Use ticket text as the RAG query
2. Retrieve top-3 relevant knowledge base chunks
3. Generate a grounded maintenance/triage response
4. Record: retrieved docs, answer, latency

**Expected outcome:** Answers reference specific SOPs (Safety tickets), equipment manuals
(Mechanical tickets), SAP guides (SAP tickets), and IT runbooks (Network tickets).


In [15]:
# CELL 6-A: Full RAG pipeline - all 40 tickets

import pandas as pd

tickets_raw = [
    {'id': 'M01', 'category': 'Mechanical',  'text': 'Pompa sentrifugal P-101 vibrasi berlebihan, perlu ganti bearing segera sebelum shutdown'},
    {'id': 'M02', 'category': 'Mechanical',  'text': 'Kompresor K-202 shutdown otomatis karena overheating, suhu mencapai 95 derajat Celsius'},
    {'id': 'M03', 'category': 'Mechanical',  'text': 'Kebocoran oli pada gearbox GB-103, seal sudah aus perlu penggantian segera'},
    {'id': 'M04', 'category': 'Mechanical',  'text': 'Motor listrik M-305 tidak bisa start, kemungkinan winding terbakar perlu megger test'},
    {'id': 'M05', 'category': 'Mechanical',  'text': 'Valve control HV-201 macet tidak bisa fully open saat startup, actuator bermasalah'},
    {'id': 'M06', 'category': 'Mechanical',  'text': 'Belt conveyor BC-101 slip dan bunyi aneh, perlu alignment dan cek kondisi belt'},
    {'id': 'M07', 'category': 'Mechanical',  'text': 'Heat exchanger HE-302 fouling parah efisiensi turun 30 persen perlu chemical cleaning'},
    {'id': 'M08', 'category': 'Mechanical',  'text': 'Pompa vacuum VP-105 tidak mencapai target vacuum ada kebocoran di flange connection'},
    {'id': 'M09', 'category': 'Mechanical',  'text': 'Coupling antara motor dan pompa P-204 rusak vibrasi tinggi perlu penggantian coupling'},
    {'id': 'M10', 'category': 'Mechanical',  'text': 'Fan cooling tower CT-101 blade retak bahaya jika dibiarkan beroperasi perlu shutdown'},
    {'id': 'S01', 'category': 'SAP/ERP',     'text': 'Error ME21N saat buat purchase order vendor master belum disetujui procurement department'},
    {'id': 'S02', 'category': 'SAP/ERP',     'text': 'Goods receipt MIGO tidak bisa diposting dokumen PO sudah closed perlu reopen PO'},
    {'id': 'S03', 'category': 'SAP/ERP',     'text': 'MIRO invoice verification gagal amount mismatch dengan PO perlu koordinasi finance'},
    {'id': 'S04', 'category': 'SAP/ERP',     'text': 'User tidak bisa akses transaction code MM01 setelah role change oleh IT admin'},
    {'id': 'S05', 'category': 'SAP/ERP',     'text': 'Batch job MB51 gagal tengah malam material document tidak ter-generate perlu rerun'},
    {'id': 'S06', 'category': 'SAP/ERP',     'text': 'Plant maintenance notification IW51 tidak bisa disimpan mandatory field kosong'},
    {'id': 'S07', 'category': 'SAP/ERP',     'text': 'SAP production order CO01 error karena bill of material tidak aktif perlu aktivasi'},
    {'id': 'S08', 'category': 'SAP/ERP',     'text': 'Report S ALR 87013019 timeout untuk period Q3 data terlalu besar perlu optimasi query'},
    {'id': 'S09', 'category': 'SAP/ERP',     'text': 'SAP login sangat lambat sejak weekend maintenance response time lebih dari 30 detik'},
    {'id': 'S10', 'category': 'SAP/ERP',     'text': 'Workflow approval purchase order stuck di inbox manager sedang cuti perlu delegate'},
    {'id': 'N01', 'category': 'Network/IT',  'text': 'SCADA server tidak bisa connect ke PLC area A network timeout perlu cek switch'},
    {'id': 'N02', 'category': 'Network/IT',  'text': 'Printer di control room offline setelah firmware update perlu rollback driver'},
    {'id': 'N03', 'category': 'Network/IT',  'text': 'WiFi area gudang intermittent operator tidak bisa scan barcode perlu cek access point'},
    {'id': 'N04', 'category': 'Network/IT',  'text': 'VPN connection ke kantor pusat putus setiap 2 jam perlu reset manual oleh IT'},
    {'id': 'N05', 'category': 'Network/IT',  'text': 'Database backup server storage penuh backup gagal 3 hari berturut-turut perlu cleanup'},
    {'id': 'N06', 'category': 'Network/IT',  'text': 'Email server bounce semua attachment lebih dari 5MB sejak kemarin perlu cek konfigurasi'},
    {'id': 'N07', 'category': 'Network/IT',  'text': 'CCTV di area produksi 4 kamera offline sekaligus kemungkinan switch port rusak'},
    {'id': 'N08', 'category': 'Network/IT',  'text': 'Server historian OSIsoft PI tidak sinkron dengan DCS data gap 6 jam perlu resync'},
    {'id': 'N09', 'category': 'Network/IT',  'text': 'Firewall block akses ke supplier portal setelah security policy update perlu whitelist'},
    {'id': 'N10', 'category': 'Network/IT',  'text': 'NTP server drift timestamp PLC berbeda 15 menit dari server SAP perlu sinkronisasi'},
    {'id': 'K01', 'category': 'Safety',      'text': 'Near miss operator hampir tertimpa material jatuh dari rak gudang perlu pasang guard'},
    {'id': 'K02', 'category': 'Safety',      'text': 'APD tidak tersedia di area welding pekerja memakai safety glasses biasa bukan welding shield'},
    {'id': 'K03', 'category': 'Safety',      'text': 'Spill oli di area pompa belum dibersihkan setelah 2 jam risiko terpeleset sangat tinggi'},
    {'id': 'K04', 'category': 'Safety',      'text': 'Izin kerja panas hot work permit tidak ditandatangani sebelum pengelasan dimulai'},
    {'id': 'K05', 'category': 'Safety',      'text': 'Prosedur LOTO belum dilakukan sebelum teknisi masuk ke dalam tangki pembersihan'},
    {'id': 'K06', 'category': 'Safety',      'text': 'Alarm kebakaran berbunyi di area laboratorium ternyata false alarm dari sensor debu'},
    {'id': 'K07', 'category': 'Safety',      'text': 'Tangga portable rusak satu anak tangga masih digunakan pekerja perlu segera diganti'},
    {'id': 'K08', 'category': 'Safety',      'text': 'Papan rambu bahaya listrik hilang di panel MCC-102 perlu pasang rambu baru segera'},
    {'id': 'K09', 'category': 'Safety',      'text': 'Pekerja kontraktor bekerja tanpa safety induction yang valid perlu hentikan pekerjaan'},
    {'id': 'K10', 'category': 'Safety',      'text': 'Tabung gas nitrogen di area proses tidak diikat ke dinding risiko jatuh dan kebocoran'},
]

df_tickets = pd.DataFrame(tickets_raw)
pipeline_results = []

print(f"Running RAG pipeline on {len(df_tickets)} tickets...")
print("-" * 60)

for _, row in df_tickets.iterrows():
    answer, retrieved, latency = rag_query(row['text'], k=3)
    top_doc = retrieved[0][0]['doc_id'] if retrieved else 'none'
    pipeline_results.append({
        'id': row['id'], 'category': row['category'],
        'top_doc': top_doc, 'latency_s': round(latency, 2),
        'answer_preview': answer[:100],
    })
    print(f"  {row['id']} [{row['category']:10s}] -> {top_doc:20s} ({latency:.2f}s)")

pipeline_df = pd.DataFrame(pipeline_results)

print(f"{'='*60}")
print(f"Pipeline complete: {len(pipeline_df)} tickets processed")
print(f"Avg latency: {pipeline_df['latency_s'].mean():.2f}s | Total: {pipeline_df['latency_s'].sum():.1f}s")
print(f"Top retrieved document by category:")
print(pipeline_df.groupby('category')['top_doc'].agg(lambda x: x.value_counts().index[0]))


Running RAG pipeline on 40 tickets...
------------------------------------------------------------


Batches: 100%|██████████| 1/1 [00:00<00:00, 76.72it/s]


  M01 [Mechanical] -> EQUIP-P101           (2.91s)


Batches: 100%|██████████| 1/1 [00:00<00:00, 288.01it/s]


  M02 [Mechanical] -> EQUIP-K202           (2.75s)


...

  K08 [Safety    ] -> SOP-ELEC-008         (2.02s)


Batches: 100%|██████████| 1/1 [00:00<00:00, 324.99it/s]


  K09 [Safety    ] -> SOP-SAFE-006         (1.97s)


Batches: 100%|██████████| 1/1 [00:00<00:00, 278.12it/s]


  K10 [Safety    ] -> EQUIP-CT101          (2.44s)
Pipeline complete: 40 tickets processed
Avg latency: 2.53s | Total: 101.2s
Top retrieved document by category:
category
Mechanical       EQUIP-K202
Network/IT        IT-RB-003
SAP/ERP       SAP-GUIDE-001
Safety         SOP-MECH-001
Name: top_doc, dtype: str


---
## Section 7 — RAG Evaluation

| Metric | Definition | How to measure |
|---|---|---|
| **Faithfulness** | Is the answer supported by retrieved context? | Answer tokens ∩ context tokens |
| **Groundedness** | Does the answer cite a source? | `[Source: ...]` pattern |
| **Relevance** | Does the answer address the question? | Query tokens ∩ answer tokens |
| **Latency** | How fast? | Wall-clock time |

### RAGAS framework (production)

In production, [RAGAS](https://github.com/explodinggradients/ragas) uses a second LLM to
evaluate answers. Tonight we implement heuristic proxies:

```python
faithfulness = len(answer_tokens & context_tokens) / len(answer_tokens)
groundedness = 1 if "[Source:" in answer else 0
relevance    = len(query_tokens & answer_tokens) / len(query_tokens)
```


In [16]:
# CELL 7-A: RAG evaluation metrics

import re

def tokenize(text):
    return set(re.findall(r'\b\w+\b', text.lower()))

STOPWORDS = {'the','a','an','is','in','of','to','and','for','that','it','this',
             'was','are','with','or','be','as','at','by','if','on','not',
             'yang','di','ke','dan','atau','ini','itu','dari','perlu','tidak'}

def faithfulness_score(answer, context_chunks):
    atoks = tokenize(answer) - STOPWORDS
    ctoks = set()
    for c in context_chunks: ctoks.update(tokenize(c))
    return len(atoks & ctoks) / len(atoks) if atoks else 0.0

def groundedness_score(answer):
    return 1.0 if re.search(r'\[Source\s*:', answer, re.IGNORECASE) else 0.0

def relevance_score(query, answer):
    qtoks = tokenize(query) - STOPWORDS
    atoks = tokenize(answer)
    return len(qtoks & atoks) / len(qtoks) if qtoks else 0.0


EVAL_IDS = ['M01', 'M02', 'S01', 'S04', 'N01', 'N04', 'K05', 'K04', 'M07', 'N08']
eval_rows = df_tickets[df_tickets['id'].isin(EVAL_IDS)].reset_index(drop=True)

eval_results = []
print("Evaluating RAG quality on 10 sample tickets...")

for _, row in eval_rows.iterrows():
    answer, retrieved, latency = rag_query(row['text'], k=3)
    ctx  = [c['text'] for c, _ in retrieved]
    docs = [c['doc_id'] for c, _ in retrieved]
    faith = faithfulness_score(answer, ctx)
    grnd  = groundedness_score(answer)
    relev = relevance_score(row['text'], answer)
    eval_results.append({
        'id': row['id'], 'category': row['category'],
        'top_doc': docs[0] if docs else 'none',
        'faithfulness': round(faith, 2), 'groundedness': round(grnd, 2),
        'relevance': round(relev, 2), 'latency_s': round(latency, 2),
    })
    print(f"  {row['id']} [{row['category']:10s}] faith={faith:.2f} "
          f"ground={grnd:.1f} relev={relev:.2f} lat={latency:.2f}s")

eval_df = pd.DataFrame(eval_results)
print(f"{'='*60}")
print("RAG Evaluation Summary")
print(f"{'='*60}")
print(eval_df[['faithfulness','groundedness','relevance','latency_s']].describe().round(2).to_string())
print(f"Mean faithfulness:  {eval_df['faithfulness'].mean():.2f}")
print(f"Mean groundedness:  {eval_df['groundedness'].mean():.2f}")
print(f"Mean relevance:     {eval_df['relevance'].mean():.2f}")
print(f"Mean latency:       {eval_df['latency_s'].mean():.2f}s")
print("Note: faithfulness < 0.5 may indicate hallucination.")
print("Note: groundedness = 0.0 means LLM did not cite its source document.")


Evaluating RAG quality on 10 sample tickets...


Batches: 100%|██████████| 1/1 [00:00<00:00, 211.66it/s]


  M01 [Mechanical] faith=0.38 ground=1.0 relev=0.36 lat=2.74s


Batches: 100%|██████████| 1/1 [00:00<00:00, 333.20it/s]


  M02 [Mechanical] faith=0.54 ground=1.0 relev=0.42 lat=5.58s


...

  N08 [Network/IT] faith=0.47 ground=1.0 relev=0.75 lat=3.35s


Batches: 100%|██████████| 1/1 [00:00<00:00, 303.98it/s]


  K04 [Safety    ] faith=0.59 ground=1.0 relev=0.30 lat=6.88s


Batches: 100%|██████████| 1/1 [00:00<00:00, 268.04it/s]


  K05 [Safety    ] faith=0.43 ground=1.0 relev=0.10 lat=2.72s
RAG Evaluation Summary
       faithfulness  groundedness  relevance  latency_s
count      10.00000      10.00000   10.00000   10.00000
mean        0.47000       1.00000    0.57000    3.33000
std         0.11000       0.00000    0.31000    1.60000
min         0.30000       1.00000    0.10000    2.10000
25%         0.39000       1.00000    0.36000    2.38000
50%         0.46000       1.00000    0.50000    2.73000
75%         0.58000       1.00000    0.87000    3.22000
max         0.63000       1.00000    1.00000    6.88000
Mean faithfulness:  0.47
Mean groundedness:  1.00
Mean relevance:     0.57
Mean latency:       3.33s
Note: faithfulness < 0.5 may indicate hallucination.
Note: groundedness = 0.0 means LLM did not cite its source document.


---
## Section 8 — Reflection & Day 4 Preview

### What we built tonight

| Step | What we did | Key concept |
|---|---|---|
| 1 | Demonstrated LLM hallucination | Parametric vs. non-parametric memory |
| 2 | Built 30-document knowledge base | Document loading & chunking |
| 3 | Encoded chunks with sentence-transformers | Dense embeddings, multilingual |
| 4 | Built FAISS and ChromaDB indexes | Vector database fundamentals |
| 5 | Implemented `rag_query()` | Context injection, prompt assembly |
| 6 | Ran all 40 tickets through pipeline | End-to-end RAG |
| 7 | Evaluated faithfulness, groundedness, latency | RAG metrics |

### Day 1 → Day 2 → Day 3 progression

| Day | Approach | Result |
|---|---|---|
| Day 1 | K-Means clustering | Silhouette = 0.027 (vocabulary overlap problem) |
| Day 2 | LLM zero-shot classification | 90%+ category accuracy |
| Day 3 | LLM + RAG | Grounded, cited answers from SOPs/manuals |

### Day 4 Preview: LangChain & LangGraph

| Topic | What it adds |
|---|---|
| LangChain RAG chain | Declarative pipeline, built-in retrievers, memory |
| LangChain document loaders | PDF, Word, HTML, S3, SharePoint → auto-chunking |
| LangGraph retrieval agent | Multi-step: retrieve → reason → re-retrieve if needed |
| RAGAS evaluation | LLM-judge faithfulness (not heuristic) |

### Homework

1. Add a **metadata filter** to ChromaDB: retrieve only `category='SOP'` for Safety tickets.
2. Implement **BM25 retrieval** (sparse) with `rank-bm25` and compare with dense.
3. Try **re-ranking**: retrieve top-10 with FAISS, re-rank with a cross-encoder model.
4. Add **streaming output** to `rag_query()` so the answer appears word-by-word.
5. Build a **Gradio UI** showing retrieved chunks + LLM answer side-by-side.

---
*Day 3 complete. See you in Day 4 — LangChain, LangGraph, and production-grade RAG.*
